# AutoLyap: Iteration-independent Lyapunov analyses - Example 2

[Open in Colab](https://colab.research.google.com/github/ManuUpadhyaya/tutorial_ELLIIT_2026/blob/main/notebooks/iteration_independent_lyapunov_example_2.ipynb)

This notebook reproduces the gradient method with constant Nesterov momentum example from the iteration-independent Lyapunov analysis slides. It is fully filled in: there are no exercise placeholders.


## Setup

Install AutoLyap. The notebook uses `backend="cvxpy"` and `cvxpy_solver="CLARABEL"`, which is suitable for Google Colab and does not require a commercial SDP solver.


In [ ]:
%pip install -q autolyap


## Problem setup

Consider the smooth convex minimization problem

$$
\underset{y\in\mathcal{H}}{\operatorname{minimize}}\quad f(y),
$$

or equivalently the inclusion problem

$$
\text{find } y\in\mathcal{H}\;\text{ such that }\;0\in\partial f(y)=\{\nabla f(y)\},
$$

where

$$
f\in\mathcal{F}_{0,L}(\mathcal{H}),\qquad 0<L<+\infty.
$$

The gradient method with constant Nesterov momentum is

$$
\begin{aligned}
y^k &= x^k+\delta(x^k-x^{k-1}),\\
x^{k+1} &= y^k-\gamma\nabla f(y^k),
\end{aligned}
$$

with $\gamma>0$ and $\delta\in\mathbb{R}$. We use AutoLyap to search for certificates that imply

$$
f(y^k)-f(y^\star)\in o(1/k)\quad\text{as}\quad k\to\infty,
$$

where $y^\star\in\operatorname{Argmin}_{y\in\mathcal{H}} f(y)$.


## State-space representation

$$
\begin{aligned}
\mathbf{x}^{k+1}
&=\left(\begin{bmatrix}1+\delta&-\delta\\1&0\end{bmatrix}\otimes I\right)\mathbf{x}^k
+\left(\begin{bmatrix}-\gamma\\0\end{bmatrix}\otimes I\right)\mathbf{u}^k,\\
\mathbf{y}^{k}
&=\left(\begin{bmatrix}1+\delta&-\delta\end{bmatrix}\otimes I\right)\mathbf{x}^k
+\left(\begin{bmatrix}0\end{bmatrix}\otimes I\right)\mathbf{u}^k,\\
\mathbf{u}^k
&\in \partial f(y_1^k)=\{\nabla f(y_1^k)\}.
\end{aligned}
$$

where

$$
\begin{aligned}
\mathbf{x}^k &= (x^k,x^{k-1}),\\
\mathbf{u}^k &= \nabla f(y^k),\\
\mathbf{y}^k &= y_1^k=y^k=x^k+\delta(x^k-x^{k-1}).
\end{aligned}
$$

The built-in `GradientNesterovMomentum` class in AutoLyap implements exactly these matrices.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

from autolyap import IterationIndependent as II
from autolyap import SolverOptions
import autolyap.algorithms as alg
import autolyap.problemclass as pc

# CLARABEL can return inaccurate statuses near the boundary of the feasible
# region. We record statuses below instead of printing the same warning many
# times during the grid sweep.
warnings.filterwarnings(
    "ignore",
    message="Solution may be inaccurate.*",
)


## Define the problem data

The slide uses $L=1$. We start with the representative momentum pair $\gamma=1$ and $\delta=0.5$.


In [ ]:
L = 1.0
gamma = 1.0
delta = 0.5

problem = pc.InclusionProblem([pc.SmoothConvex(L=L)])
solver_options = SolverOptions(backend="cvxpy", cvxpy_solver="CLARABEL")


## Sublinear Lyapunov certificate

For this example, AutoLyap chooses $P,p,T,t$ so that the lower bound in **C2** is zero:

$$
\left\langle \boldsymbol{\xi}^{k}_{\mathrm{vec}},
(P\otimes I)\boldsymbol{\xi}^{k}_{\mathrm{vec}}\right\rangle
+p^{\top}\boldsymbol{\xi}^{k}_{\mathrm{func}}=0,
$$

and the residual lower bound in **C3** is function-value suboptimality:

$$
\left\langle \boldsymbol{\xi}^{k}_{\mathrm{vec}},
(T\otimes I)\boldsymbol{\xi}^{k}_{\mathrm{vec}}\right\rangle
+t^{\top}\boldsymbol{\xi}^{k}_{\mathrm{func}}
=f(y^k)-f(y^\star).
$$

We set $\rho=1$ and keep **C4**, the monotonicity condition for the residual. The resulting certificate implies last-iterate convergence

$$
f(y^k)-f(y^\star)\in o(1/k).
$$


In [ ]:
def search_sublinear_certificate(gamma_value: float, delta_value: float):
    algorithm = alg.GradientNesterovMomentum(
        gamma=float(gamma_value),
        delta=float(delta_value),
    )
    P, p, T, t = II.SublinearConvergence.get_parameters_function_value_suboptimality(
        algorithm,
    )
    return II.search_lyapunov(
        problem,
        algorithm,
        P,
        T,
        p=p,
        t=t,
        rho=1.0,
        remove_C4=False,
        solver_options=solver_options,
        verbosity=0,
    )


single_result = search_sublinear_certificate(gamma, delta)
print(f"gamma: {gamma:g}")
print(f"delta: {delta:g}")
print(f"status: {single_result['status']}")
print(f"solver status: {single_result['solve_status']}")


## Sweep over momentum parameters

The slide visualizes feasible pairs $(\gamma,\delta)$ for $L=1$. The grid below is intentionally moderate so that the notebook remains practical in Google Colab. Increase `num_gamma_values` or `num_delta_values` for a smoother boundary.


In [ ]:
num_gamma_values = 45
num_delta_values = 45

gamma_values = np.linspace(0.05, 3.5, num_gamma_values)
delta_values = np.linspace(-0.98, 0.98, num_delta_values)
zero_delta_index = int(np.argmin(np.abs(delta_values)))
if abs(delta_values[zero_delta_index]) < 1e-12:
    delta_values[zero_delta_index] = 0.0

feasible_points = []
solve_statuses = []

for delta_index, delta_value in enumerate(delta_values, start=1):
    row_feasible_count = 0
    for gamma_value in gamma_values:
        try:
            result = search_sublinear_certificate(gamma_value, delta_value)
            status = result["status"]
            solve_status = result["solve_status"]
        except Exception as exc:
            status = "error"
            solve_status = type(exc).__name__
        solve_statuses.append((status, solve_status))
        if status == "feasible":
            feasible_points.append((gamma_value, delta_value))
            row_feasible_count += 1
    print(
        f"delta row {delta_index:02d}/{num_delta_values}: "
        f"delta={delta_value:+.3f}, feasible={row_feasible_count}/{num_gamma_values}",
        flush=True,
    )

feasible_points = np.array(feasible_points, dtype=float)
unique_statuses = sorted(set(solve_statuses))

print(
    f"feasible grid points: {len(feasible_points)} / "
    f"{num_gamma_values * num_delta_values}"
)
print("statuses:")
for status, solve_status in unique_statuses:
    count = solve_statuses.count((status, solve_status))
    print(f"  {status:10s} {str(solve_status):22s} {count}")


In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 4.2), dpi=160, constrained_layout=True)

if feasible_points.size:
    ax.scatter(
        feasible_points[:, 0],
        feasible_points[:, 1],
        s=16,
        color="#4f8bc9",
        edgecolors="none",
        label="AutoLyap feasible",
        zorder=3,
    )

ax.set_axisbelow(True)
ax.set_xlabel(r"$\gamma$")
ax.set_ylabel(r"$\delta$", rotation=0)
ax.set_title(r"Gradient method with constant Nesterov momentum")
ax.set_xlim(0.0, 3.5)
ax.set_ylim(-1.0, 1.0)
ax.set_xticks([0, 1, 2, 3])
ax.set_yticks([-1, -0.5, 0, 0.5, 1])
ax.grid(True, alpha=0.3, zorder=0)
if feasible_points.size:
    ax.legend(loc="lower right")
plt.show()
